<a href="https://colab.research.google.com/github/davidrpugh/data-analytics-with-polars/blob/main/ch12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 12: Working with Textual, Temporal, and Nested Data Types

This notebook accompanies Chapter 12 of *Python Polars: The Definitive Guide* by Jeroen Janssens and Thijs Nieuwdorp. It is adapted for **Polars 2.1** and **Google Colab**.

Some data types deserve special attention because they have their own **namespace** of methods (Chapter 7). This chapter covers three groups:

| Group | Data types | Namespace(s) |
|---|---|---|
| **Textual** | `String`, `Categorical`, `Enum` | `.str`, `.cat` |
| **Temporal** | `Date`, `Datetime`, `Time`, `Duration` | `.dt` |
| **Nested** | `List`, `Array`, `Struct` | `.list`, `.arr`, `.struct` |

By the end of this notebook you will be able to:

- clean, search, split, and parse **text**
- choose between `String`, `Categorical`, and `Enum`, and know **how categoricals changed in Polars 2.x**
- parse, format, and compute with **dates, datetimes, durations, and time zones**
- store and process **lists**, fixed-size **arrays**, and **structs**

**How to use this notebook:** run the cells from top to bottom. Some cells *intentionally* trigger errors or warnings.

## 0. Setup

We pin Polars to the **2.1 series** so everyone sees the same behavior.

In [ ]:
%pip install -q "polars==2.1.*"

### Imports and helpers

- `show_warnings` runs code and prints every **warning** it triggers, in addition to any error. That's how we'll *see* deprecations (such as the string cache) rather than read about them.
- `download` fetches a data file from the book's repository **only if it isn't already there**, writes to a temporary name and renames when complete, and raises on HTTP errors.

In [ ]:
from datetime import date, datetime
from pathlib import Path
import urllib.request
import warnings

import polars as pl
from polars.testing import assert_frame_equal


BASE_URL = "https://github.com/davidrpugh/data-analytics-with-polars/raw/refs/heads/main/data"
DATA_DIR = Path("sample_data")


def download(relative_path: str) -> Path:
    """Download BASE_URL/relative_path into DATA_DIR (unless it exists) and return the local path."""
    path = DATA_DIR / relative_path
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(path.name + ".tmp")
        urllib.request.urlretrieve(f"{BASE_URL}/{relative_path}", temporary)
        temporary.rename(path)
    return path

In [ ]:
print(f"Polars {pl.__version__}")

## String

A **String** is text: a sequence of characters, digits, or symbols. Polars stores it as **UTF-8**, where one character takes 1 to 4 bytes. All the String methods live in the `.str` namespace, and fall into three groups:

| Group | What for | Examples |
|---|---|---|
| **Conversion** | to and from other types | `to_date`, `to_datetime`, `to_integer`, `strptime`, `json_decode`, `encode`/`decode` |
| **Describing and querying** | ask questions about the text | `contains`, `starts_with`, `ends_with`, `len_chars`, `len_bytes`, `count_matches`, `find` |
| **Manipulation** | change the text | `strip_chars`, `to_lowercase`, `replace`, `replace_all`, `slice`, `split`, `extract`, `extract_all`, `pad_start`, `zfill` |

### Cleaning text

Let's start with some messy text and clean it in **one chain**: strip surrounding whitespace, lowercase it, and replace underscores with spaces. Reading the chain from top to bottom tells the story of what happens to each value.

In [ ]:
corpus = pl.DataFrame(
    {
        "raw_text": [
            "  Data Science is amazing ",
            "Data_analysis > Data entry",
            " Python&Polars; Fast",
        ]
    }
)

corpus = corpus.with_columns(
    processed_text=pl.col("raw_text")
    .str.strip_chars()
    .str.to_lowercase()
    .str.replace_all("_", " ")
)

corpus

### Slicing and splitting

`str.slice(offset, length)` cuts a piece out by **position**. `str.split(" ")` cuts the text at every space and returns a **List** of the parts (more on lists below), from which `list.get(i)` picks one.

Notice that `list.get` fails for a position that doesn't exist. A text with only two words has no third word. We show what your version does:

In [ ]:
corpus.select(
    "processed_text",
    first_5_chars=pl.col("processed_text").str.slice(0, 5),
    first_word=pl.col("processed_text").str.split(" ").list.get(0),
    second_word=pl.col("processed_text").str.split(" ").list.get(1),
)

In [ ]:
corpus.select(fifth_word=pl.col("processed_text").str.split(" ").list.get(4))

If a word may be missing, `list.get` has an option to return `null` instead of failing (check the documentation for `null_on_oob`). When you want just the *first* word, a regular expression with `str.extract` is a more direct tool: it returns the text matched by a **capture group** (the part in parentheses).

In [ ]:
corpus.select(
    "processed_text",
    first_word=pl.col("processed_text").str.extract(r"^(\S+)"),
    fifth_word_or_null=pl.col("processed_text").str.split(" ").list.get(4, null_on_oob=True),
)

### Length: characters or bytes?

"How long is this text?" has two answers. `len_chars()` counts **characters**; `len_bytes()` counts the **bytes** of the UTF-8 encoding. For plain English letters they agree. For other characters they don't (an accented letter takes 2 bytes, a Japanese character 3, an emoji 4).

Use `len_chars()` when you mean *how many letters*. `len_bytes()` is much **faster**, since the byte count is known without scanning the text, so use it when you only need a size, or when all the text is ASCII.

In [ ]:
pl.DataFrame(
    {
        "text": ["naive", "naïve", "日本語", "😀"]
    }
).select(
    "text",
    chars=pl.col("text").str.len_chars(),
    bytes=pl.col("text").str.len_bytes(),
)

### Searching: regular expressions and literals

Many `str` methods (`contains`, `replace`, `replace_all`, `count_matches`, `extract`, ...) treat their pattern as a **regular expression** by default. That's powerful, but it means characters like `.`, `+`, and `*` have special meanings. Pass `literal=True` when you mean plain text.

`count_matches` counts how often a pattern occurs and `contains("a.b")` as a regex matches *any* character between `a` and `b`:

In [ ]:
corpus.select(
    "processed_text",
    count_a=pl.col("processed_text").str.count_matches("a"),
)

In [ ]:
pl.DataFrame(
    {
        "s": ["a.b", "axb"]
    }
).select(
    "s",
    as_regex=pl.col("s").str.contains("a.b"),
    as_literal=pl.col("s").str.contains("a.b", literal=True),
)

A pattern that is not a valid regex raises an error. For instance, a lone `+` has nothing to repeat. With `literal=True` the same call works:

In [ ]:
plus_df = pl.DataFrame({"s": ["1+1", "2+2"]})

plus_df.select(pl.col("s").str.replace_all("+", " plus "))

In [ ]:
plus_df.select(pl.col("s").str.replace_all("+", " plus ", literal=True))

### Extracting matches with regular expressions

`str.extract_all` returns **every** match as a List. This regex finds hashtags: `#` followed by one or more word characters (`\w+`: letters, digits, underscore). A post without hashtags gets an empty list.

In [ ]:
posts = pl.DataFrame(
    {
        "post": ["Loving #python and #polars!", "A boomer post without a hashtag"]
    }
)

hashtag_regex = r"#(\w+)"

posts.with_columns(hashtags=pl.col("post").str.extract_all(hashtag_regex))

### Converting text to other types

Text columns often hold numbers and dates in disguise. `str.to_integer(base=...)` interprets text in a number base (for instance hexadecimal), and `str.to_date(format)` parses dates (Chapter 12 covers the format codes in the temporal section).

Both are **strict** by default: a value that can't be converted raises an error. Pass `strict=False` to get `null` instead.

In [ ]:
pl.select(pl.lit("ff").str.to_integer(base=16))

In [ ]:
bad_dates = pl.DataFrame({"text": ["2024-02-29", "2023-02-29", "not a date"]})

bad_dates.select(pl.col("text").str.to_date("%Y-%m-%d"))

In [ ]:
parsed = bad_dates.with_columns(date=pl.col("text").str.to_date("%Y-%m-%d", strict=False))

parsed

## Categorical and Enum

When a text column has **few distinct values repeated many times** (countries, sizes, statuses), storing every string separately wastes memory and makes comparisons slow. A **Categorical** stores each distinct string **once**, in a dictionary, and the column itself holds small integer codes:

- the integer is the **physical** representation
- the string is the **lexical** representation

`to_physical()` shows the integer codes:

In [ ]:
cats = pl.DataFrame(
    {
        "name": ["Persian cat", "Siamese Cat", "Lynx", "Lynx"]
    },
    schema={
        "name": pl.Categorical
    },
)

cats.with_columns(name_physical=pl.col("name").to_physical())

### What changed in Polars 2.x

The book describes an older design, in which **each Categorical column had its own private dictionary**. Two columns with the same strings in a different order would have different integer codes, so combining them required an expensive "re-encoding" and raised a `CategoricalRemappingWarning`. The remedy was the **string cache** (`pl.StringCache()` or `pl.enable_string_cache()`), a dictionary shared by all columns.

In Polars 2.x, categoricals are backed by **`pl.Categories`**, a named collection of categories. By default, **all categoricals share one global set of categories**. That has visible consequences, and you can verify them in the next cells.

**1. The same string gets the same code in different DataFrames.** Compare with the table above. `Lynx` and `Siamese Cat` get the same codes, and a new string (`Maine Coon Cat`) simply gets the next free code. (The exact numbers depend on which categoricals were created earlier in your session, so **never rely on the physical codes**.)

In [ ]:
more_cats = pl.DataFrame(
    {"name": ["Maine Coon Cat", "Lynx", "Lynx", "Siamese Cat"]},
    schema={"name": pl.Categorical},
)

more_cats.with_columns(name_physical=pl.col("name").to_physical())

**2. Joining two categorical columns just works**, with no re-encoding and no warning. The book shows a `CategoricalRemappingWarning` here.

In [ ]:
joined = cats.join(more_cats, on="name")

joined

**3. The string cache is deprecated.** The old tools still run, but they tell you they have been replaced:

In [ ]:
with pl.StringCache():
    df = pl.DataFrame({"x": ["a", "b"]}, schema={"x": pl.Categorical})

df

**4. `cat.get_categories()` was removed in Polars 2.0.** The error message points to the replacements:

- for a *Categorical*, the distinct values **present in a column** are just `unique()`
- for an *Enum*, the fixed list of categories is `dtype.categories`

In [ ]:
cats.select(pl.col("name").cat.get_categories())

In [ ]:
cats.select(pl.col("name").unique(maintain_order=True))

**5. Sorting is always lexical.** In older versions you could choose between *physical* and *lexical* ordering (`pl.Categorical("physical")`). The "physical" option is deprecated, and categoricals now sort **alphabetically by their string value**, whichever option you pass. Let's confirm that both give the same alphabetical result, and see the deprecation warning:

In [ ]:
sorted_default = cats.sort("name")
sorted_physical = cats.with_columns(pl.col("name").cast(pl.Categorical("physical"))).sort("name")


In [ ]:
sorted_default

In [ ]:
sorted_physical

**Summary of the changes:**

| Book (Polars 1.x) | Polars 2.1 |
|---|---|
| Each Categorical has a private dictionary | All Categoricals share one global set of categories (`pl.Categories`) |
| Joining different Categoricals warns and re-encodes | Joins just work |
| `pl.StringCache()`, `pl.enable_string_cache()` | Deprecated: replaced by `pl.Categories` |
| `cat.get_categories()` | Removed: use `unique()` (Categorical) or `dtype.categories` (Enum) |
| Sort order: physical (default) or lexical | Always lexical |


### Enum: when the categories are known in advance

If you know the possible values **up front**, use an **Enum**. The categories are fixed when you create the type. That has three advantages over a Categorical:

1. **Validation**: a value outside the list is an error
2. **Meaningful order**: values sort in the order you declared, not alphabetically
3. **Self-describing**: `dtype.categories` lists the allowed values

In [ ]:
bear_enum_dtype = pl.Enum(["Polar", "Panda", "Brown"])

bear_enum_series = pl.Series(
    "bear", ["Polar", "Panda", "Brown", "Brown", "Polar"], dtype=bear_enum_dtype
)
bear_cat_series = pl.Series(
    "bear", ["Polar", "Panda", "Brown", "Brown", "Polar"], dtype=pl.Categorical
)

print("Enum categories:", bear_enum_dtype.categories.to_list())
print("Enum sorted:       ", bear_enum_series.sort().to_list())
print("Categorical sorted:", bear_cat_series.sort().to_list())

The Enum sorts in the declared order (Polar, Panda, Brown), the Categorical alphabetically (Brown, Panda, Polar). A value outside the declared list is rejected:

In [ ]:
pl.Series("bear", ["Polar", "Grizzly"], dtype=bear_enum_dtype)

Because an Enum has a known order, **comparisons** between values make sense (is "Panda" before "Brown" in the declared order?).

In [ ]:
(bear_enum_series < "Brown")

### How much memory do they save?

Let's measure instead of guessing. A million values with only three distinct strings:

In [ ]:
values = ["Polar", "Panda", "Brown"] * 333_334
as_string = pl.Series("bear", values)
as_categorical = as_string.cast(pl.Categorical)
as_enum = as_string.cast(bear_enum_dtype)

pl.DataFrame(
    {
        "type": ["String", "Categorical", "Enum"],
        "size_mb": [round(s.estimated_size("mb"), 2) for s in (as_string, as_categorical, as_enum)],
    }
)

**Which one?**

| Use | When |
|---|---|
| `String` | the values are mostly unique (names, free text), or you do lots of text manipulation |
| `Categorical` | few distinct values, which you don't know in advance (or that change) |
| `Enum` | few distinct values that you **do** know in advance, especially when order matters or you want validation |

Enums don't have their own method namespace; you work with them using the general-purpose methods (and `.cat` where it applies).

## Temporal

Temporal data types describe **points and intervals in time**. They support comparison, arithmetic, and time-specific operations through the `.dt` namespace.

| Type | Meaning | Stored as |
|---|---|---|
| `Date` | a calendar date (no time of day) | days since 1970-01-01 |
| `Datetime` | a date and a time of day (with a time unit, and optionally a time zone) | time since 1970-01-01 (default microseconds) |
| `Time` | a time of day only | nanoseconds since midnight |
| `Duration` | the difference between two points in time (like Python's `timedelta`) | a count in a time unit |

The `.dt` methods fall into groups: **conversion** (`to_string`, `strftime`, `cast_time_unit`), **describing** (`year`, `month`, `weekday`, `hour`, `total_days`, `is_leap_year`, ...), and **manipulation** (`truncate`, `round`, `offset_by`, `month_start`, `month_end`, `convert_time_zone`, `replace_time_zone`, ...).

### Loading dates from a CSV file

A CSV file has no types, so dates arrive as text unless you say otherwise. You have two options:

- `try_parse_dates=True`: Polars **guesses** which columns contain dates
- `schema_overrides={"date": pl.Date}`: you **state** the type (more predictable, and an error if the text doesn't fit)

For anything important, prefer stating the type. We check that both agree here:

In [ ]:
ALL_STOCKS_CSV = download("all_stocks.csv")

stocks = pl.read_csv(ALL_STOCKS_CSV, try_parse_dates=True)
stocks_explicit = pl.read_csv(ALL_STOCKS_CSV, schema_overrides={"date": pl.Date})

In [ ]:
assert_frame_equal(stocks, stocks_explicit)

In [ ]:
print(stocks.schema)

In [ ]:
stocks

### Converting to and from text

`str.to_date(format)` parses text, and `dt.to_string(format)` writes a date as text. The format uses the codes of Rust's *chrono* library (`%Y` year, `%m` month, `%d` day, `%H` hour, `%M` minute, `%S` second, `%z` UTC offset, and so on; they mostly match Python's `strftime`).

In [ ]:
dates = pl.DataFrame({"date_str": ["2023-12-31", "2024-02-29"]}).with_columns(
    date=pl.col("date_str").str.to_date("%Y-%m-%d")
)

dates.with_columns(formatted_date=pl.col("date").dt.to_string("%d-%m-%Y"))

### Pulling dates apart, and computing with them

The `.dt` namespace extracts parts (`year`, `month`, `weekday`, `quarter`, ...) and rolls dates forward or back. Notes: `weekday()` follows the ISO convention, **Monday = 1** to **Sunday = 7**. `offset_by("1mo")` moves by a calendar month (so 31 January + 1 month is the end of February, not an error). `month_end()` rolls forward to the last day of the month.

In [ ]:
pl.DataFrame({"date": [date(2024, 1, 31), date(2024, 2, 29), date(2024, 12, 31)]}).with_columns(
    year=pl.col("date").dt.year(),
    month=pl.col("date").dt.month(),
    weekday=pl.col("date").dt.weekday(),
    quarter=pl.col("date").dt.quarter(),
    is_leap_year=pl.col("date").dt.is_leap_year(),
    plus_one_month=pl.col("date").dt.offset_by("1mo"),
    month_end=pl.col("date").dt.month_end(),
)

Subtracting two dates gives a **Duration**, which you can turn into a number with `total_days()`, `total_hours()`, and similar methods:

In [ ]:
pl.DataFrame({"start": [date(2024, 1, 1)], "end": [date(2024, 3, 1)]}).with_columns(
    duration=pl.col("end") - pl.col("start")
).with_columns(days=pl.col("duration").dt.total_days())

Extracting parts is how you **group by time**. Here, average closing price per company per year. (In Chapter 13 we do this with `group_by_dynamic`, which is built for time windows.)

In [ ]:
(
    stocks.group_by("symbol", year=pl.col("date").dt.year())
          .agg(avg_close=pl.col("close").mean().round(2))
          .sort("symbol", "year")
)

### Generating date ranges

`pl.date_range(start, end, interval)` creates a sequence of dates. The `interval` is text such as `"1d"` (a day), `"1w"` (a week), `"1h"` (an hour), or `"1mo"` (a month). It is an **expression**, so inside a `select` it just works. Inside the `DataFrame(...)` constructor you must ask for the result as a Series with `eager=True`, because a constructor can't take an expression:

In [ ]:
pl.DataFrame(
    {
        "monday": pl.date_range(
            start=pl.date(2024, 10, 28),
            end=pl.date(2024, 12, 1),
            interval="1w",
            eager=True,
        ),
    }
)

In [ ]:
pl.DataFrame({"monday": pl.date_range(pl.date(2024, 10, 28), pl.date(2024, 12, 1), "1w")})

### Time zones

Time zones are the most unpleasant part of working with time, and daylight saving time is the worst of it. The practical advice: **store and compute in UTC**, then convert to a local time zone only for display.

There are two operations, and mixing them up is a classic bug:

- `replace_time_zone(tz)` **labels** a wall-clock time with a zone without changing the clock reading. It gives a *different moment in time*. Use it when you know which zone a "naive" timestamp was recorded in.
- `convert_time_zone(tz)` **translates** a zone-aware timestamp to another zone. The *moment* stays the same, and only the clock reading changes.

We parse text with mixed UTC offsets (note the `%z` in the format, which reads the offset), which yields UTC datetimes, and convert them to Amsterdam time. Look at the offset change on 28 March 2021, when Amsterdam switched from winter time (CET) to summer time (CEST):

In [ ]:
pl.DataFrame(
    {
        "utc_mixed_offset": [
            "2021-03-27T00:00:00+0100",
            "2021-03-28T00:00:00+0100",
            "2021-03-29T00:00:00+0200",
            "2021-03-30T00:00:00+0200",
        ]
    }
).with_columns(
    parsed=pl.col("utc_mixed_offset").str.to_datetime("%Y-%m-%dT%H:%M:%S%z")
).with_columns(
    converted=pl.col("parsed").dt.convert_time_zone("Europe/Amsterdam")
)

Now the difference between *replace* and *convert*, on one timestamp (noon, no time zone attached). Replacing the zone keeps the clock at 12:00 but changes the moment. Converting keeps the moment, and changes the clock:

In [ ]:
naive = pl.DataFrame({"naive": [datetime(2024, 7, 1, 12, 0)]})

naive.with_columns(
    replaced=pl.col("naive").dt.replace_time_zone("Europe/Amsterdam"),
    in_utc=pl.col("naive").dt.replace_time_zone("UTC"),
).with_columns(
    utc_converted_to_amsterdam=pl.col("in_utc").dt.convert_time_zone("Europe/Amsterdam"),
)

**Daylight saving gaps.** When clocks jump forward, some local times **don't exist**: in Amsterdam, 02:30 on 31 March 2024 never happened. Asking Polars to attach a time zone to such a time is an error by default (a wrong answer would be worse).

In [ ]:
gap = pl.DataFrame({"naive": [datetime(2024, 3, 31, 2, 30)]})

gap.select(pl.col("naive").dt.replace_time_zone("Europe/Amsterdam"))

In [ ]:
gap.select(pl.col("naive").dt.replace_time_zone("Europe/Amsterdam", non_existent="null"))

## List

A **List** column holds a *sequence of values of the same type* in each row, and the sequences **can have different lengths**. All List methods live in the `.list` namespace.

> **Not Python's `list`.** A Python list can mix types; a Polars List can't, and it is stored in an efficient columnar layout. You *can* put Python lists in a column of type `Object`, but that gives up all the optimizations, so avoid it.

Lists appear often as the *result* of other operations, such as `str.split` or `extract_all` (above), or `implode`. Common methods: `len`, `get`, `first`, `last`, `contains`, `join` (for lists of strings), `sort`, `unique`, `sum`/`mean`/`min`/`max`, `eval`, and `explode`.

### Evaluating Booleans in a list

`list.all()` and `list.any()` reduce a list of Booleans to one Boolean per row:

In [ ]:
bools = pl.DataFrame({"values": [[True, True], [False, False, True], [False]]})

bools.with_columns(
    all_true=pl.col("values").list.all(),
    any_true=pl.col("values").list.any(),
)

### Running an expression on every element: `list.eval`

`list.eval` applies **any expression** to the *elements* of each list, where `pl.element()` stands for "the current element". This is how you transform the inside of a list without leaving Polars, which is much faster than calling a Python function for each value.

Here we test which ages are over forty, then use `list.all()` to ask whether *everybody* in the group is. The `parallel=True` argument lets the expression run in parallel (off by default). It can speed up expensive expressions. Notice that the second step needs a **separate `with_columns`**, because expressions in one call can't use each other's results (Chapter 10).

In [ ]:
groups = pl.DataFrame({"ages": [[18, 21], [30, 40, 50], [42, 69]]})

groups.with_columns(
    over_forty=pl.col("ages").list.eval(pl.element() > 40, parallel=True)
).with_columns(
    all_over_forty=pl.col("over_forty").list.all()
)

### Summaries, searching, and sorting

Many list methods **summarize each list** to one value, without any `eval`:

In [ ]:
groups.with_columns(
    n=pl.col("ages").list.len(),
    oldest=pl.col("ages").list.max(),
    mean_age=pl.col("ages").list.mean(),
    has_40=pl.col("ages").list.contains(40),
    first_age=pl.col("ages").list.first(),
)

In [ ]:
groups.with_columns(ages_sorted_descending=pl.col("ages").list.sort(descending=True))

For lists of **strings**, `list.join` glues the parts back together, which pairs nicely with `str.split` (split, change some parts, join):

In [ ]:
pl.DataFrame({"words": [["data", "science"], ["polars"]]}).with_columns(
    sentence=pl.col("words").list.join(" ")
)

### Exploding a list into rows

`explode` turns **each element of a list into its own row**. The other columns are repeated for every new row. It exists both as a DataFrame method and as an expression (`list.explode()`, with the alias `flatten()`). Chapter 15 covers it in depth.

In [ ]:
groups.explode("ages")

In [ ]:
groups.select(ages=pl.col("ages").list.explode())

## Array

An **Array** is like a List whose length is **fixed**: every row has the same number of elements (like a NumPy array's shape). Because the length is known, Polars needs no offset buffer to find each row's values, so Arrays are **more compact and faster** than Lists. Use an Array for fixed-size data such as coordinates, embeddings, or **one week of daily readings**. The `.arr` namespace has methods like `max`, `median`, `sum`, `arg_max`, `get`, `contains`, `sort`, and conversion with `to_list` and `to_struct`.

We create a table of daily temperatures: one Array of seven values (Monday to Sunday) per city.

In [ ]:
events = pl.DataFrame(
    [
        pl.Series("location", ["Paris", "Amsterdam", "Barcelona"], dtype=pl.String),
        pl.Series(
            "temperatures",
            [
                [23, 27, 21, 22, 24, 23, 22],
                [17, 19, 15, 22, 18, 20, 21],
                [30, 32, 28, 29, 34, 33, 31],
            ],
            dtype=pl.Array(pl.Int64, shape=7),
        ),
    ]
)

events

`arg_max` returns the **position** of the maximum (counting from 0), which is the weekday index here (0 = Monday). We translate it to a name with `replace_strict`, which, unlike `replace`, fails if a value has no mapping (Chapter 8).

In [ ]:
weekdays = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"}

events.with_columns(
    median=pl.col("temperatures").arr.median(),
    max=pl.col("temperatures").arr.max(),
    warmest_dow=pl.col("temperatures").arr.arg_max(),
).with_columns(
    warmest_day=pl.col("warmest_dow").replace_strict(weekdays, return_dtype=pl.String)
)

**The length is enforced.** An Array column refuses rows of a different length, which is what makes it a good place to put "this must be exactly seven values":

In [ ]:
pl.Series("temperatures", [[1, 2, 3, 4, 5, 6, 7], [1, 2, 3]], dtype=pl.Array(pl.Int64, shape=7))

You can convert between the two: `arr.to_list()` always works; `list.to_array(width)` works when every list has that length:

In [ ]:
as_list = events.select("location", temperatures=pl.col("temperatures").arr.to_list())
print(as_list.schema)

In [ ]:
as_array = as_list.select("location", temperatures=pl.col("temperatures").list.to_array(7))
print(as_array.schema)

In [ ]:
assert_frame_equal(as_array, events.select("location", "temperatures"))

## Struct

A **Struct** stores **several named fields** in one column. Think of each row as a Python dictionary: the keys are the **fields**, and each field has its own data type. A Struct is how Polars lets *one* expression work on *several* columns at once, so it is central to the idiomatic Polars style. Building a Struct from existing columns doesn't copy the data. The Struct just points to it.

The `.struct` namespace has `field` (get one field), `unnest` (expand all fields into columns), `rename_fields`, `with_fields`, and `json_encode`.

### Creating a Struct

A Struct is created from a list of Python dictionaries:

In [ ]:
orders = pl.DataFrame(
    {
        "customer_id": [2781, 6139, 5392],
        "order_details": [
            {"amount": 250.00, "date": date(2024, 1, 3), "items": 5},
            {"amount": 150.00, "date": date(2024, 1, 5), "items": 1},
            {"amount": 100.00, "date": date(2024, 1, 2), "items": 3},
        ],
    },
)

In [ ]:
print(orders.schema)


In [ ]:
orders

### Getting fields out

`struct.field(name)` picks one field as an ordinary column. `unnest` (a DataFrame method, and also available as `struct.unnest()` on an expression) expands **all** fields into columns:

In [ ]:
orders.select(pl.col("order_details").struct.field("amount"))

In [ ]:
order_details_df = orders.unnest("order_details")
order_details_df

### Putting fields in

The opposite operation, **combining columns into a Struct**, is `pl.struct`. Here we select the three columns again and add a Struct that bundles them.

In [ ]:
nested_again = order_details_df.select(
    "customer_id",
    order_details=pl.struct("amount", "date", "items"),
)

nested_again

### Changing a Struct

`rename_fields`, `with_fields`, and `json_encode` change a Struct without unnesting it.

In [ ]:
orders.select(pl.col("order_details").struct.rename_fields(["total", "when", "n_items"]))

In [ ]:
orders.select(
    pl.col("order_details").struct.with_fields(amount_with_tax=pl.field("amount") * 1.21)
).unnest("order_details")

In [ ]:
orders.select(pl.col("order_details").struct.json_encode())

### A common source of Structs: `value_counts`

Some methods *return* Structs. The best known is `value_counts`, which counts how often each distinct value occurs. It returns a Struct with two fields: the original column name and `count`. Unnest it to get an ordinary two-column table. (Ties between equal counts can come out in any order.)

In [ ]:
basket = pl.DataFrame(
    {"fruit": ["cherry", "apple", "banana", "banana", "apple", "banana"]}
)

basket.select(pl.col("fruit").value_counts(sort=True))

In [ ]:
basket.select(pl.col("fruit").value_counts(sort=True).struct.unnest())

## Takeaways

- Each special type has its own **namespace**: `.str`, `.cat`, `.dt`, `.list`, `.arr`, `.struct`. Enums have none.
- **Strings** are UTF-8. `len_chars` counts characters, `len_bytes` counts bytes (faster). Patterns are **regular expressions** unless you pass `literal=True`. String conversions are **strict**; use `strict=False` and count the nulls.
- **Categoricals** in Polars 2.x share **one global set of categories** (`pl.Categories`), so joins just work, sorting is always lexical, `pl.StringCache` is deprecated, and `cat.get_categories()` is gone (use `unique()` or `dtype.categories`). Never rely on the physical codes.
- Use an **Enum** when the categories are known in advance: it validates, has a meaningful order, and is compact.
- **Dates** from CSV files: state the type with `schema_overrides` rather than relying on guessing. Subtracting dates gives a **Duration**. `weekday()` is ISO (Monday = 1).
- **Time zones**: compute in UTC. `replace_time_zone` *labels*, `convert_time_zone` *translates*. A local time that doesn't exist (a daylight-saving gap) is an error by default.
- **Lists** have variable length; **Arrays** have fixed length (faster and enforced). `list.eval` with `pl.element()` runs an expression on the elements. `explode` turns elements into rows.
- A **Struct** bundles named fields into one column. `unnest` and `pl.struct` convert in both directions.
- When in doubt about a version-specific behavior: **run the code and look**. The `try_it` and `show_warnings` helpers make that easy.

### Next up

In Chapter 13 we **summarize and aggregate**: `group_by`, window functions, and dynamic (time-based) group-bys.

### Try it yourself


1. Write one expression chain that turns `"  The QUICK brown Fox  "` into `"the-quick-brown-fox"` (strip, lowercase, then replace spaces with hyphens).

2. Count the hashtags in each post of `posts` with `str.extract_all` and `list.len`. Which post has the most?

3. Make an `Enum` of the days of the week and sort a small column of weekday names. Compare with sorting the same names as plain strings.

4. Create the datetime `2024-10-27 02:30` in Amsterdam time. Careful: on that date, clocks go *back*, so this time occurs **twice**. Check the `ambiguous` argument of `replace_time_zone`.

5. Use `str.split(",")` on a column of comma-separated tags, then `list.explode()`, then `value_counts` to find the most common tag.

6. Convert the `order_details` struct into JSON text, then parse it back with `str.json_decode`. Do you get the same struct?